In [22]:
import pandas as pd
import numpy as np

industry = pd.read_csv("../outputs/cleaned_industry.csv")

syllabus = pd.read_csv("../outputs/cleaned_syllabus.csv")
syllabus["Institute"] = syllabus["Institute"].str.lower().str.strip()

print("Industry shape:", industry.shape)
print("Industry columns:", industry.columns.tolist())
print("Syllabus institutes:", syllabus["Institute"].unique())
industry.head()

Industry shape: (710, 10)
Industry columns: ['Company', 'Year', 'Role', 'Skill', 'Skill_Type', 'Upcoming_Flag', 'Taught_Institute_Count', 'Is_Taught_MSRIT', 'Institutional_Gap_Score', 'Skill_Domain']
Syllabus institutes: ['rvce' 'bit' 'pesit' 'dsce' 'msrit']


,Company,Year,Role,Skill,Skill_Type,Upcoming_Flag,Taught_Institute_Count,Is_Taught_MSRIT,Institutional_Gap_Score,Skill_Domain
0,Oracle,2021,Backend Developer,java,Traditional,0,4,1,0.2,Other
1,Oracle,2021,Backend Developer,spring boot,Transitional,0,0,0,1.0,Other
2,Oracle,2021,Backend Developer,sql,Transitional,0,5,1,0.0,Data
3,Oracle,2021,Software Engineer,java,Traditional,0,4,1,0.2,Other
4,Oracle,2021,Software Engineer,data structures,Transitional,0,1,1,0.8,Data


In [23]:
freq = industry.groupby("Skill").size().reset_index(name="Freq")
freq.head()

,Skill,Freq
0,advanced sql,8
1,api gateway,1
2,api rate limiting,1
3,auto scaling,1
4,aws,29


In [24]:
from sklearn.linear_model import LinearRegression

def compute_slope(skill_df):
    yearly = skill_df.groupby("Year").size().reset_index(name="Count")
    if len(yearly) > 1:
        X = yearly["Year"].values.reshape(-1, 1)
        y = yearly["Count"].values
        model = LinearRegression().fit(X, y)
        return model.coef_[0]
    else:
        return 0

trend = (
    industry.groupby("Skill")
    .apply(compute_slope, include_groups=False)
    .reset_index(name="Trend_Slope")
)
trend.head()

,Skill,Trend_Slope
0,advanced sql,-4.000000
1,api gateway,0.000000
2,api rate limiting,0.000000
3,auto scaling,0.000000
4,aws,-1.571429


In [25]:
max_year = industry["Year"].max()
min_year = industry["Year"].min()

industry["Recency"] = (industry["Year"] - min_year) / (max_year - min_year)

recency = industry.groupby("Skill")["Recency"].mean().reset_index(name="Recency_Weight")

print("Recency_Weight range:", recency["Recency_Weight"].min().round(3), "to", recency["Recency_Weight"].max().round(3))
recency.head()

Recency_Weight range: 0.0 to 1.0


,Skill,Recency_Weight
0,advanced sql,0.812500
1,api gateway,0.250000
2,api rate limiting,1.000000
3,auto scaling,0.500000
4,aws,0.318966


In [26]:
upcoming = industry.groupby("Skill")["Upcoming_Flag"].max().reset_index()
upcoming.head()

,Skill,Upcoming_Flag
0,advanced sql,1
1,api gateway,0
2,api rate limiting,0
3,auto scaling,0
4,aws,0


In [27]:
features = freq.merge(trend, on='Skill')
features = features.merge(recency, on='Skill')
features = features.merge(upcoming, on='Skill')
print('Base features shape:', features.shape)
features.head()

Base features shape: (95, 5)


,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag
0,advanced sql,8,-4.000000,0.812500,1
1,api gateway,1,0.000000,0.250000,0
2,api rate limiting,1,0.000000,1.000000,0
3,auto scaling,1,0.000000,0.500000,0
4,aws,29,-1.571429,0.318966,0


In [28]:
print("Is_Taught_MSRIT distribution:")
print(industry["Is_Taught_MSRIT"].value_counts())
print()
print("Skills taught at MSRIT:", industry[industry['Is_Taught_MSRIT']==1]['Skill'].nunique())
print("Skills NOT taught at MSRIT:", industry[industry['Is_Taught_MSRIT']==0]['Skill'].nunique())

Is_Taught_MSRIT distribution:
Is_Taught_MSRIT
1    467
0    243
Name: count, dtype: int64

Skills taught at MSRIT: 43
Skills NOT taught at MSRIT: 52


In [29]:
print("Taught_Institute_Count distribution:")
print(industry["Taught_Institute_Count"].value_counts().sort_index())
features = freq.merge(trend, on="Skill")
features = features.merge(recency, on="Skill")
features = features.merge(upcoming, on="Skill")

taught_cols = industry.drop_duplicates('Skill')[['Skill','Is_Taught_MSRIT','Taught_Institute_Count','Institutional_Gap_Score']]
features = features.merge(taught_cols, on='Skill', how='left')
features['Is_Taught_MSRIT'] = features['Is_Taught_MSRIT'].fillna(0).astype(int)
features['Taught_Institute_Count'] = features['Taught_Institute_Count'].fillna(0).astype(int)
features['Institutional_Gap_Score'] = features['Institutional_Gap_Score'].fillna(1.0)
print()
print('Features shape so far:', features.shape)
features.head()

Taught_Institute_Count distribution:
Taught_Institute_Count
0    174
1    135
2     98
3     35
4    176
5     92
Name: count, dtype: int64

Features shape so far: (95, 8)


,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag,Is_Taught_MSRIT,Taught_Institute_Count,Institutional_Gap_Score
0,advanced sql,8,-4.000000,0.812500,1,1,3,0.4
1,api gateway,1,0.000000,0.250000,0,1,1,0.8
2,api rate limiting,1,0.000000,1.000000,0,0,0,1.0
3,auto scaling,1,0.000000,0.500000,0,0,0,1.0
4,aws,29,-1.571429,0.318966,0,0,0,1.0


In [30]:
numeric_cols = ["Freq", "Trend_Slope", "Recency_Weight", "Upcoming_Flag", "Taught_Institute_Count"]
corr = features[numeric_cols].corr().round(2)
print("Feature correlation matrix:")
print(corr)

Feature correlation matrix:
                        Freq  Trend_Slope  Recency_Weight  Upcoming_Flag  \
Freq                    1.00        -0.27           -0.08          -0.07   
Trend_Slope            -0.27         1.00           -0.10          -0.16   
Recency_Weight         -0.08        -0.10            1.00           0.58   
Upcoming_Flag          -0.07        -0.16            0.58           1.00   
Taught_Institute_Count  0.57        -0.29           -0.11          -0.09   

                        Taught_Institute_Count  
Freq                                      0.57  
Trend_Slope                              -0.29  
Recency_Weight                           -0.11  
Upcoming_Flag                            -0.09  
Taught_Institute_Count                    1.00  


In [31]:
TOTAL_COMPANIES = industry['Company'].nunique()
print(f'Total companies: {TOTAL_COMPANIES}')

company_spread = (
    industry.groupby('Skill')['Company']
    .nunique()
    .reset_index(name='Company_Spread')
)

company_spread['Company_Spread_Norm'] = (company_spread['Company_Spread'] / TOTAL_COMPANIES).round(3)

print('Company_Spread distribution:')
print(company_spread['Company_Spread'].value_counts().sort_index())
company_spread.head()

Total companies: 10
Company_Spread distribution:
Company_Spread
1     51
3      2
4      2
5      1
6      4
7      6
8      4
9     13
10    12
Name: count, dtype: int64


,Skill,Company_Spread,Company_Spread_Norm
0,advanced sql,8,0.8
1,api gateway,1,0.1
2,api rate limiting,1,0.1
3,auto scaling,1,0.1
4,aws,10,1.0


In [32]:
def compute_velocity(skill_df):
    yearly = (
        skill_df.groupby("Year")
        .size()
        .reset_index(name="Count")
        .sort_values("Year")
    )
    if len(yearly) < 2:
        return 0.0
    deltas = yearly["Count"].diff().dropna()
    return deltas.mean()

velocity = (
    industry.groupby("Skill")
    .apply(compute_velocity, include_groups=False)
    .reset_index(name="Velocity")
)

print("Velocity stats:")
print(velocity["Velocity"].describe().round(3))
print("\nTop 5 fastest growing skills:")
print(velocity.sort_values("Velocity", ascending=False).head())
print("\nTop 5 fastest declining skills:")
print(velocity.sort_values("Velocity").head())

Velocity stats:
count    95.000
mean     -0.558
std       1.927
min      -8.000
25%       0.000
50%       0.000
75%       0.000
max       8.000
Name: Velocity, dtype: float64

Top 5 fastest growing skills:
                  Skill  Velocity
85           tensorflow       8.0
18   data visualization       2.5
21  distributed systems       1.5
40           kubernetes       1.0
47     machine learning       1.0

Top 5 fastest declining skills:
                 Skill  Velocity
27       generative ai      -8.0
15      cloud security      -7.0
43                 llm      -7.0
36                java      -5.0
13  cloud architecture      -4.0


In [33]:
def compute_volatility(skill_df):
    yearly = skill_df.groupby("Year").size()
    
    all_years = range(industry["Year"].min(), industry["Year"].max() + 1)
    yearly = yearly.reindex(all_years, fill_value=0)
    return yearly.std()

volatility = (
    industry.groupby("Skill")
    .apply(compute_volatility, include_groups=False)
    .reset_index(name="Volatility")
)

print("Volatility stats:")
print(volatility["Volatility"].describe().round(3))
print("\nMost volatile skills (hype candidates):")
print(volatility.sort_values("Volatility", ascending=False).head())
print("\nMost stable skills (consistent demand):")
print(volatility.sort_values("Volatility").head())

Volatility stats:
count    95.000
mean      2.080
std       2.252
min       0.447
25%       0.447
50%       0.447
75%       3.194
max      11.524
Name: Volatility, dtype: float64

Most volatile skills (hype candidates):
         Skill  Volatility
78         sql   11.523888
40  kubernetes   10.535654
85  tensorflow    6.655825
36        java    6.348228
22      docker    5.813777

Most stable skills (consistent demand):
                        Skill  Volatility
2           api rate limiting    0.447214
7   background job processing    0.447214
9           canary deployment    0.447214
8       blue-green deployment    0.447214
14    cloud cost optimization    0.447214


In [34]:
features = features.merge(company_spread, on='Skill', how='left')
features = features.merge(velocity, on='Skill', how='left')
features = features.merge(volatility, on='Skill', how='left')

features['Company_Spread'] = features['Company_Spread'].fillna(1).astype(int)
features['Company_Spread_Norm'] = features['Company_Spread_Norm'].fillna(0.1).round(3)
features['Velocity'] = features['Velocity'].fillna(0.0)
features['Volatility'] = features['Volatility'].fillna(0.0)

print('Final feature table shape:', features.shape)
print('Columns:', list(features.columns))
features.head(10)

Final feature table shape: (95, 12)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility']


,Skill,Freq,Trend_Slope,Recency_Weight,Upcoming_Flag,Is_Taught_MSRIT,Taught_Institute_Count,Institutional_Gap_Score,Company_Spread,Company_Spread_Norm,Velocity,Volatility
0,advanced sql,8,-4.000000,0.812500,1,1,3,0.4,8,0.8,-4.0,2.607681
1,api gateway,1,0.000000,0.250000,0,1,1,0.8,1,0.1,0.0,0.447214
2,api rate limiting,1,0.000000,1.000000,0,0,0,1.0,1,0.1,0.0,0.447214
3,auto scaling,1,0.000000,0.500000,0,0,0,1.0,1,0.1,0.0,0.447214
4,aws,29,-1.571429,0.318966,0,0,0,1.0,10,1.0,-2.0,4.494441
5,aws ec2,1,0.000000,0.000000,0,0,0,1.0,1,0.1,0.0,0.447214
6,azure,3,0.000000,0.250000,0,1,2,0.6,3,0.3,0.0,1.341641
7,background job processing,1,0.000000,1.000000,0,0,0,1.0,1,0.1,0.0,0.447214
8,blue-green deployment,1,0.000000,1.000000,0,0,0,1.0,1,0.1,0.0,0.447214
9,canary deployment,1,0.000000,1.000000,0,0,0,1.0,1,0.1,0.0,0.447214


In [35]:
all_numeric = [
    "Freq", "Trend_Slope", "Recency_Weight", "Upcoming_Flag",
    "Taught_Institute_Count", "Company_Spread", "Velocity", "Volatility"
]
corr = features[all_numeric].corr().round(2)
print("Updated correlation matrix:")
print(corr)

Updated correlation matrix:
                        Freq  Trend_Slope  Recency_Weight  Upcoming_Flag  \
Freq                    1.00        -0.27           -0.08          -0.07   
Trend_Slope            -0.27         1.00           -0.10          -0.16   
Recency_Weight         -0.08        -0.10            1.00           0.58   
Upcoming_Flag          -0.07        -0.16            0.58           1.00   
Taught_Institute_Count  0.57        -0.29           -0.11          -0.09   
Company_Spread          0.86        -0.35           -0.15          -0.11   
Velocity               -0.29         0.99           -0.10          -0.16   
Volatility              0.92        -0.25           -0.14          -0.06   

                        Taught_Institute_Count  Company_Spread  Velocity  \
Freq                                      0.57            0.86     -0.29   
Trend_Slope                              -0.29           -0.35      0.99   
Recency_Weight                           -0.11           -0

In [36]:
skill_type_map = industry.groupby('Skill')['Skill_Type'].agg(lambda x: x.mode()[0]).reset_index()
check = features.merge(skill_type_map, on='Skill', how='left')

print('Mean features by Skill_Type:')
print(
    check.groupby('Skill_Type')[
        ['Freq','Trend_Slope','Velocity','Volatility','Company_Spread','Recency_Weight']
    ].mean().round(3)
)
print()
print('NOTE: Upcoming skills should ideally show positive Trend_Slope and Velocity.')
print('If not, some Upcoming labels may be based on recency of first appearance')
print('rather than sustained growth — this is expected in a small 5-year dataset.')

Mean features by Skill_Type:
                Freq  Trend_Slope  Velocity  Volatility  Company_Spread  \
Skill_Type                                                                
Traditional   11.778       -0.660    -0.750       2.110           4.889   
Transitional   7.549       -0.196    -0.235       2.382           4.686   
Upcoming       4.346       -1.041    -1.058       1.465           3.000   

              Recency_Weight  
Skill_Type                    
Traditional            0.346  
Transitional           0.346  
Upcoming               0.763  

NOTE: Upcoming skills should ideally show positive Trend_Slope and Velocity.
If not, some Upcoming labels may be based on recency of first appearance
rather than sustained growth — this is expected in a small 5-year dataset.


In [ ]:
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "--quiet"], check=False)

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

unique_skills = features["Skill"].tolist()
print(f"Encoding {len(unique_skills)} skills...")
skill_vecs = embedder.encode(unique_skills, show_progress_bar=True, batch_size=64)

all_centroid = skill_vecs.mean(axis=0, keepdims=True)
centrality = cosine_similarity(skill_vecs, all_centroid).flatten()

msrit_taught_mask = features["Is_Taught_MSRIT"].values == 1
if msrit_taught_mask.sum() > 0:
    msrit_centroid = skill_vecs[msrit_taught_mask].mean(axis=0, keepdims=True)
    gap_distance = 1 - cosine_similarity(skill_vecs, msrit_centroid).flatten()
else:
    gap_distance = np.zeros(len(unique_skills))

features["Semantic_Centrality"]    = centrality.round(4)
features["Semantic_Gap_Distance"]  = gap_distance.round(4)

print("Semantic feature stats:")
print(features[["Semantic_Centrality", "Semantic_Gap_Distance"]].describe().round(4))
print("Top 5 semantically central skills (core tech):")
print(features.nlargest(5, "Semantic_Centrality")[["Skill", "Semantic_Centrality"]])
print("Top 5 semantically distant gap skills (niche/novel):")
print(features[features["Is_Taught_MSRIT"]==0].nlargest(5, "Semantic_Gap_Distance")[["Skill", "Semantic_Gap_Distance"]])


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1110.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 95 skills...


Batches: 100%|██████████| 2/2 [00:00<00:00,  2.07it/s]

Semantic feature stats:
       Semantic_Centrality  Semantic_Gap_Distance
count              95.0000                95.0000
mean                0.3933                 0.6198
std                 0.0996                 0.1155
min                 0.1509                 0.3423
25%                 0.3157                 0.5409
50%                 0.4053                 0.5943
75%                 0.4582                 0.7075
max                 0.6318                 0.9127
Top 5 semantically central skills (core tech):
                  Skill  Semantic_Centrality
47     machine learning               0.6318
13   cloud architecture               0.5765
21  distributed systems               0.5716
4                   aws               0.5493
54          multi-cloud               0.5413
Top 5 semantically distant gap skills (niche/novel):
                Skill  Semantic_Gap_Distance
66        react query                 0.9127
91  vpc configuration                 0.8384
89         typescript

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
tfidf_matrix = tfidf.fit_transform(unique_skills)

rarity = np.array(tfidf_matrix.mean(axis=1)).flatten()
features["NLP_Rarity_Score"] = rarity.round(5)

print("TF-IDF rarity stats:")
print(features["NLP_Rarity_Score"].describe().round(5))
print("5 most common-vocabulary skills (broad, foundational terms):")
print(features.nsmallest(5, "NLP_Rarity_Score")[["Skill", "NLP_Rarity_Score"]])
print("5 most rare-vocabulary skills (niche, emerging tech terms):")
print(features.nlargest(5, "NLP_Rarity_Score")[["Skill", "NLP_Rarity_Score"]])


TF-IDF rarity stats:
count    95.00000
mean      0.00693
std       0.00195
min       0.00431
25%       0.00431
50%       0.00746
75%       0.00747
max       0.01140
Name: NLP_Rarity_Score, dtype: float64
5 most common-vocabulary skills (broad, foundational terms):
     Skill  NLP_Rarity_Score
4      aws           0.00431
6    azure           0.00431
16     css           0.00431
22  docker           0.00431
24   excel           0.00431
5 most rare-vocabulary skills (niche, emerging tech terms):
                                   Skill  NLP_Rarity_Score
69  retrieval augmented generation (rag)           0.01140
7              background job processing           0.00964
12               circuit breaker pattern           0.00964
20            disaster recovery planning           0.00964
35                infrastructure as code           0.00964


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

N_CLUSTERS = 8
normed_vecs = normalize(skill_vecs)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(normed_vecs)

features["Skill_Cluster"] = cluster_labels

print(f"K-Means clusters (k={N_CLUSTERS}) — representative skills per cluster:")
for c in range(N_CLUSTERS):
    members = features[features["Skill_Cluster"] == c]["Skill"].tolist()
    print(f"  Cluster {c}: {members[:6]}")

cluster_dummies = pd.get_dummies(features["Skill_Cluster"], prefix="Cluster")
features = pd.concat([features, cluster_dummies], axis=1)
print(f"Cluster one-hot columns added: {list(cluster_dummies.columns)}")


K-Means clusters (k=8) — representative skills per cluster:
  Cluster 0: ['aws', 'aws ec2', 'azure', 'cloud architecture', 'cloud cost optimization', 'cloud security']
  Cluster 1: ['auto scaling', 'background job processing', 'css', 'data structures', 'data visualization', 'deep learning']
  Cluster 2: ['disaster recovery planning', 'event-driven architecture', 'golang', 'infrastructure as code', 'lighthouse audits', 'observability']
  Cluster 3: ['api gateway', 'api rate limiting', 'jwt authentication', 'llm apis', 'oauth2', 'rest apis']
  Cluster 4: ['javascript', 'next.js', 'node.js', 'react', 'react query', 'typescript']
  Cluster 5: ['advanced sql', 'sql', 'sql optimization', 'sql window functions']
  Cluster 6: ['blue-green deployment', 'canary deployment', 'github actions', 'model deployment', 'model drift detection', 'model monitoring']
  Cluster 7: ['ci/cd', 'ci/cd pipelines', 'circuit breaker pattern', 'grpc', 'power bi', 'vpc configuration']
Cluster one-hot columns added: [

In [ ]:
if "Skill_Domain" in industry.columns:
    domain_map = (
        industry.drop_duplicates("Skill")[["Skill", "Skill_Domain"]]
    )
    features = features.merge(domain_map, on="Skill", how="left")
    features["Skill_Domain"] = features["Skill_Domain"].fillna("Other")
    domain_dummies = pd.get_dummies(features["Skill_Domain"], prefix="Domain")
    features = pd.concat([features, domain_dummies], axis=1)
    print("Skill_Domain distribution in feature table:")
    print(features["Skill_Domain"].value_counts())
else:
    print("Skill_Domain column not found — re-run 01_data_cleaning.ipynb with NLP enabled")


Skill_Domain distribution in feature table:
Skill_Domain
Other            47
Cloud            12
Data              9
AI/ML             8
Web/Mobile        7
DevOps            6
Security          4
Systems/Infra     2
Name: count, dtype: int64


In [ ]:
features.to_csv("../outputs/feature_table.csv", index=False)
print("Saved feature_table.csv")
print("Shape:", features.shape)
print("Columns:", list(features.columns))
print()

nlp_cols = [c for c in features.columns if c.startswith(("Semantic_", "NLP_", "Cluster_", "Domain_", "Skill_Cluster", "Skill_Domain"))]
print(f"NLP-derived columns ({len(nlp_cols)}): {nlp_cols}")
nulls = features.isnull().sum()
if nulls.sum() == 0:
    print("No null values - clean!")
else:
    print("WARNING - nulls found:")
    print(nulls[nulls>0])


Saved feature_table.csv
Shape: (95, 33)
Columns: ['Skill', 'Freq', 'Trend_Slope', 'Recency_Weight', 'Upcoming_Flag', 'Is_Taught_MSRIT', 'Taught_Institute_Count', 'Institutional_Gap_Score', 'Company_Spread', 'Company_Spread_Norm', 'Velocity', 'Volatility', 'Semantic_Centrality', 'Semantic_Gap_Distance', 'NLP_Rarity_Score', 'Skill_Cluster', 'Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6', 'Cluster_7', 'Skill_Domain', 'Domain_AI/ML', 'Domain_Cloud', 'Domain_Data', 'Domain_DevOps', 'Domain_Other', 'Domain_Security', 'Domain_Systems/Infra', 'Domain_Web/Mobile']

NLP-derived columns (21): ['Semantic_Centrality', 'Semantic_Gap_Distance', 'NLP_Rarity_Score', 'Skill_Cluster', 'Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4', 'Cluster_5', 'Cluster_6', 'Cluster_7', 'Skill_Domain', 'Domain_AI/ML', 'Domain_Cloud', 'Domain_Data', 'Domain_DevOps', 'Domain_Other', 'Domain_Security', 'Domain_Systems/Infra', 'Domain_Web/Mobile']
No null values - c